# Imports

In [ ]:
import sys
from pathlib import Path
import string
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as patches
  
if Path.cwd().name in ['ETL', 'data_science']:
    root = Path.cwd().parent
else:
    root = Path.cwd()

if str(root) not in sys.path:
    sys.path.append(str(root))
else:
    None
from utils import align_working_dir, save_chart

align_working_dir('data_science') 

%run ./0_content_load.ipynb

In [ ]:
# Save charts to a specific folder with a given prefix in the name.
CHART_PREFIX = "event_"

# Annual Thematic Evolution by Categories
- **Clustered Bar Chart**

In [ ]:
event_mix_data = combined_df[combined_df['Type'] == 'Event'].copy()

# Create Pivot Table and Normalize to 100%
# Rows = Year, Columns = Category
pivot_df = event_mix_data.groupby(['Year',
'Category']).size().unstack(fill_value=0)
pivot_pct = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

category_totals = pivot_df.sum()  # Total count per event category
total_events = int(category_totals.sum())  # Grand total of all events

# --- Plotting ---
ax = pivot_pct.plot(kind='bar', stacked=True, figsize=(12,8),
colormap='tab20', width=0.8, rot=0)

fig = ax.get_figure()

fig.suptitle('Events: Annual Thematic Evolution', x=0.5, y=0.98, ha='center')
ax.set_title(f'n={total_events}')

ax.set_ylabel('Percentage of Total Events (%)')
ax.set_xlabel('Year')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

legend_labels = [f'{col} (n={int(category_totals[col])})' for col in
pivot_pct.columns]

ax.legend(
    legend_labels,
    title='Event Categories:',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    frameon=False,
    alignment='left'
)

# Add percentage labels inside the bars
for p in ax.patches:
    h = p.get_height()
    if h > 5: # Only label if there's enough space (at least 5% height)
        ax.annotate(f'{h:.0f}%',
                    (p.get_x() + p.get_width()/2., p.get_y() + h/2.),
                    ha='center', va='center', fontsize=16, color='white',
fontweight='medium')
        
# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_thematic_evolution", CHART_PREFIX)
plt.show()

# Event Performance Overview 

- **Horizontal Lollipop Chart** with two subplots

In [ ]:
# Filter for 'Event' Type
events_only = combined_df[combined_df['Type'] == 'Event'].copy()

# Comprehensive Mapping for all 13 Categories
mapping = {
    'simontonklub': 'Simonton klub',
    'simontonösszehangolva': 'Összehangolva a Simonton módszerrel',
    'egyéb': 'Egyéb\n(kirándulás, karácsonyi party, stb.)',
    'jólneveldasárkányod': 'Jól neveled a Sárkányod!',
    'esetmegbeszélés': 'Esetmegbeszélés',
    'erőforrásnap': 'Erőforrás nap',
    'irodalomterápia': 'Irodalomterápia Simonton alapokon',
    'meditációgyógyítóerő': 'Meditáció, mint gyógyító erő',
    'gyorrsegély': 'Gyorssegély daganatos betegeknek',
    'vankiútaszorongásból': 'Van kiút a szorongásból!',
    'együttegymásért': 'Együtt - Egymásért',
    'erőforrástábor': 'Erőforrás tábor',
    'mseterápia': 'Meseterápia útkeresőknek'
}

# Clean categories and filter to ensure we only include mapped ones
events_only['Category'] = events_only['Category'].astype(str).str.strip().str.lower()
events_only = events_only[events_only['Category'].isin(mapping.keys())]

# Aggregate Data (Total for ALL time)
event_stats = events_only.groupby('Category').agg(
    Total_Events=('Category', 'count'),
    Total_Guests=('Guest_Going', 'sum')
).sort_values('Total_Events', ascending=False).reset_index()

# Assign Labels (A, B, C...)
def get_label(i):
    return string.ascii_uppercase[i] if i < 26 else f"A{string.ascii_uppercase[i-26]}"

event_stats['Label'] = [get_label(i) for i in range(len(event_stats))]

# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10), sharey=True)

# 1. Create formatted Y-axis labels (Category + Description in parentheses)
y_labels = [f"{mapping[row['Category']]}" for _, row in event_stats.iterrows()]

# 2. Plot ax1 (Events)
ax1.hlines(y=range(len(event_stats)), xmin=0, xmax=event_stats['Total_Events'],
color=CUSTOM_PALETTE['Event'], alpha=0.5)
ax1.scatter(event_stats['Total_Events'], range(len(event_stats)),
color=CUSTOM_PALETTE['Event'], s=100)

# 3. Plot ax2 (Guests)
ax2.hlines(y=range(len(event_stats)), xmin=0, xmax=event_stats['Total_Guests'],
color='#4361EE', alpha=0.5)
ax2.scatter(event_stats['Total_Guests'], range(len(event_stats)),
color='#4361EE', s=100)


ax1.set_yticks(range(len(event_stats)))
ax1.set_yticklabels(y_labels, fontsize=10)

ax1.invert_yaxis() # Highest count at the top
ax1.set_title("Frequency of Events", fontweight='bold', pad=15)
ax2.set_title("Total Attendance", fontweight='bold', pad=15)


for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', left=True, bottom=True)

fig.suptitle('Event Performance Overview', x=0.1, y=0.98, ha='left')
fig.text(0.1, 0.90, 'Which Events Drive the Most Engagement?\nTotal aggregated performance | Period: 2019-2025', ha='left', fontsize=14, linespacing=1.5, alpha=0.7)

plt.tight_layout(rect=[0, 0, 1, 0.93]) # Automated spacing that respects titles

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "lollipop_engagement_overview", CHART_PREFIX)
plt.show()

- **Horizontal Bar Chart** with two subplots

In [ ]:
# --- Plotting ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 10), sharey=True)

# 1. Create formatted Y-axis labels
y_labels = [f"{mapping[row['Category']]}" for _, row in event_stats.iterrows()]

# 2. Plot ax1 (Events) - Switched to BARH
bars1 = ax1.barh(range(len(event_stats)), event_stats['Total_Events'], 
                 color=CUSTOM_PALETTE['Event'], alpha=0.8, height=0.7)
ax1.bar_label(bars1, padding=8, fontweight='bold', alpha=0.7)

# 3. Plot ax2 (Guests) - Switched to BARH
bars2 = ax2.barh(range(len(event_stats)), event_stats['Total_Guests'], 
                 color='#4361EE', alpha=0.8, height=0.7)
ax2.bar_label(bars2, padding=8, fontweight='bold', alpha=0.7)

# Formatting
ax1.set_yticks(range(len(event_stats)))
ax1.set_yticklabels(y_labels, fontsize=10)

ax1.invert_yaxis() # Highest count at the top
ax1.set_title("Frequency of Events", fontweight='bold', pad=15)
ax2.set_title("Total Attendance", fontweight='bold', pad=15)

for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.tick_params(axis='both', which='major', left=True, bottom=True)
    # Add a subtle grid for the x-axis to help read bar lengths
    ax.xaxis.grid(True, linestyle='--', alpha=0.3)

fig.suptitle('Event Performance Overview', x=0.1, y=0.98, ha='left', fontsize=18, fontweight='bold')
fig.text(0.1, 0.9, 'Which Events Drive the Most Engagement?\nTotal aggregated performance | Period: 2019-2025', 
         ha='left', fontsize=14, linespacing=1.5, alpha=0.7)

plt.tight_layout(rect=[0, 0, 1, 0.93])

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "bar_engagement_overview", CHART_PREFIX)
plt.show()

# The "Waitlist-to-Going" Ratio (Supply vs. Demand)
  Create a simple **Bar Chart** showing the ratio of Waitlist / Going per category.
   * The Goal: Identify which categories are "Under-served." If a category has a high ratio, it means you
     need to host more of those events because people are being turned away.


In [ ]:
# Aggregate both Going and Waitlist counts
ratio_df = events_only.groupby('Category').agg(
    Total_Going=('Guest_Going', 'sum'),
    Total_Waitlist=('Guest_Waitlist', 'sum')
).reset_index()

# Calculate the Ratio (Waitlist / Going)
# We add a tiny epsilon (1e-6) or handle zeros to avoid division by zero errors
ratio_df['Waitlist_Ratio'] = ratio_df['Total_Waitlist'] / ratio_df['Total_Going']

# 3. Map the descriptions and sort by the highest ratio
ratio_df['Description'] = ratio_df['Category'].map(mapping)
ratio_df = ratio_df.sort_values('Waitlist_Ratio', ascending=False)

# --- Plotting ---
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(
    data=ratio_df,
    x='Waitlist_Ratio',
    y='Description',
    color='#e7a67e', 
    ax=ax
)

for p in ax.patches:
       width = p.get_width()
       ax.annotate(f'{width:.1%}',
                   (width, p.get_y() + p.get_height() / 2.),
                   ha='left', va='center', xytext=(5, 0),
                   textcoords='offset points', fontsize=11, fontweight='bold')

fig.suptitle('Unmet Demand: Waitlist-to-Going Ratio', x=0.5, y=0.98, ha='center')
ax.set_title('Categories with high ratios (e.g., >10%) indicate \na strong need for more frequent sessions or larger venues.', loc='left', pad=20, alpha=0.8)

ax.set_xlabel('Ratio (Waitlist Guests / Going Guests)')
ax.set_ylabel('')

# Add a vertical "Threshold" line at 10% for visual reference
ax.axvline(0.1, color='red', linestyle='--', alpha=0.5)
ax.text(0.12, 8, '10% Capacity Warning', transform=ax.get_yaxis_transform(), color='red', alpha=0.7)

# Clean up axes
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(axis='both', which='major', left=True, bottom=True)

save_chart(fig, "unmet_demand_ratio", CHART_PREFIX)
plt.show()


# Annual distribution of Guest_Going and Guest_Waitlist
- **Clustered Bar Chart**

[NOTE]  
  - **X-axis**: Years (e.g., 2023, 2024, 2025).
  - **Y-axis**: Total count of going gests and waitlist guest for all events in a given year.
  - **Details**: For each year, a separate column for Guest_Going and Guest_Waitlist.

In [ ]:
# Filter for Events and aggregate by Year
annual_guest_demand_df =  (
       combined_df.query("Type == 'Event'")
       .groupby('Year')[['Guest_Going', 'Guest_Waitlist']]
       .sum()
       .reset_index()
   )

# Reshape to "Long Format" (Tidy Data) &  Clean up labels for the legend
annual_guest_demand_melted_df = (
       annual_guest_demand_df
       .rename(columns={'Guest_Going': 'Going', 'Guest_Waitlist': 'Waitlist'})
       .melt(id_vars='Year', var_name='Guest_Status', value_name='Total_Count')
)

display(annual_guest_demand_df.head(2))
display(annual_guest_demand_melted_df.head(2))

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))


demand_palette = [CUSTOM_PALETTE['Event'], '#797f70']

sns.barplot(
    data=annual_guest_demand_melted_df,
    x='Year', 
    y='Total_Count', 
    hue='Guest_Status', 
    palette=demand_palette
)

fig.suptitle('Annual Event Demand') 

ax.set_ylabel('Total Guests')
ax.set_xlabel('Year')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Get handles to keep colors, then update labels with totals
handles, _ = ax.get_legend_handles_labels()
ax.legend(
       handles=handles,
       labels=['Going', 'Waitlist'],
       title='Guest Status:',
       loc='lower center',
       bbox_to_anchor=(0.5, 1.05),       
       ncol=2,
       frameon=False, 
   )

plt.show()

## Monthly Event Demand
- **Line Chart**

[NOTE]  
Seasonality and Trends.
   * **Insight**: "In which months do we see the highest demand for events?"
   * Why use it: It clearly shows if there are specific peaks (e.g., "Our summer events always have huge waitlists").

In [ ]:
monthly_guest_demand_df = (
        combined_df.query("Type == 'Event'")
       .groupby('Month')[['Guest_Going', 'Guest_Waitlist']]
       .sum()
       .reset_index()
    )

monthly_guest_demand_melted_df = (
       monthly_guest_demand_df
       .rename(columns={'Guest_Going': 'Going', 'Guest_Waitlist': 'Waitlist'})
       .melt(id_vars='Month', var_name='Guest_Status', value_name='Total_Count')
)

display(monthly_guest_demand_df.head(2))
display(monthly_guest_demand_melted_df.head(2))

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

demand_palette = [CUSTOM_PALETTE['Event'], '#797f70']

month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
   'Oct', 'Nov', 'Dec']

sns.lineplot(
    data=monthly_guest_demand_melted_df,
    x='Month', 
    y='Total_Count', 
    hue='Guest_Status', 
    palette=demand_palette,
    marker='o',
    markersize=8,
    linewidth=1
)

fig.suptitle('Monthly Event Demand') 
ax.set_title('Total monthly count of Going vs. Waitlisted guests\n summed across 2019-2025')

ax.set_ylabel('Total Guests')
ax.set_xlabel('Month')

# Set the positions and then the labels
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_labels)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

ax.legend(
    title='Guest Status:',
    bbox_to_anchor=(0.05, 1),
    loc='upper left', # The 'anchor point' of the legend box itself
)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "monthly_demand", CHART_PREFIX)
plt.show()

# The Distribution 
- **Box Chart** (or box-and-whisker plot)  

[NOTE]  
   * Best for: Predictability and Event Scaling.
   * Insight: "What is a 'typical' event size? Are waitlists common or rare
     outliers?"
   * Why use it: It shows the "Health" of your event planning. If the median is far
     below the mean, it means a few "super-events" are skewing your totals.

In [ ]:
# Use the un-aggregated event data to see the spread
event_dist_df = (
    combined_df.query("Type == 'Event'")[['Guest_Going', 'Guest_Waitlist']]
    .rename(columns={'Guest_Going': 'Going', 'Guest_Waitlist': 'Waitlist'})
    .melt(var_name='Guest_Status', value_name='Total_Count')
)

# --- Plotting ---
fig, ax = plt.subplots(figsize=(10, 6))

sns.boxplot(
    data=event_dist_df,
    x='Guest_Status',
    y='Total_Count',
    hue='Guest_Status', 
    palette=[CUSTOM_PALETTE['Event'], '#797f70'],
    showmeans=True,
       meanprops={"marker":"s", "markersize":'10',"markerfacecolor":"white",
   "markeredgecolor":"black", "markersize":"8"},
       medianprops={"color": "#c92638", "linewidth": 2.5}
)

fig.suptitle('Distribution of Event Attendance')
ax.set_title('Comparing typical "Going" vs "Waitlist" counts per event\n(White Square = Mean, Central Line = Median)')

ax.set_ylabel('Number of Guests')
ax.set_xlabel('Guest Status')

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "attendance_distribution", CHART_PREFIX)
plt.show()

# Suggested Code for "Speaker Influence"
  Since you run %run ./0_content_load.ipynb at the start, you have access to the full
  events_df.

In [ ]:
speaker_stats = events_df.query("date.dt.year != 2026").copy()

# Aggregating by Speaker
speaker_stats = events_df.groupby('speaker').agg(
    Avg_Going=('guest_going', 'mean'),
    Avg_Waitlist=('guest_waitlist', 'mean'),
    Event_Count=('id', 'count') # <--- We count the events here
).reset_index().sort_values('Avg_Going', ascending=False)

# --- Plotting ---
fig, ax1 = plt.subplots(figsize=(14, 7))

# --- Axis 1: Average Attendance (Bars) ---
sns.barplot(data=speaker_stats, x='speaker', y='Avg_Going', color='#4cc9f0', ax=ax1,
alpha=0.7, label='Avg Attendance')

ax1.set_ylabel('Average Attendees', color='#4cc9f0')
ax1.tick_params(axis='y', labelcolor='#4cc9f0')

# --- Axis 2: Average Waitlist (Line) ---
ax2 = ax1.twinx()
sns.lineplot(data=speaker_stats, x=range(len(speaker_stats)), y='Avg_Waitlist',marker='o', color='#f72585', linewidth=3, markersize=10, ax=ax2, label='Avg Waitlist')
ax2.set_ylabel('Average Waitlist Count', color='#f72585', fontweight='bold')
ax2.tick_params(axis='y', labelcolor='#f72585')

# 3. --- ADDING EVENT COUNT LABELS ---
for i, count in enumerate(speaker_stats['Event_Count']):
    ax1.annotate(f'n={count}',
                 (i, speaker_stats['Avg_Going'].iloc[i]),
                 textcoords="offset points",
                 xytext=(0,10),
                 ha='center',
                 fontsize=10,
                 fontweight='bold',
                 color='#0077b6')
    
fig.suptitle('Speaker Pull-Power Analysis')
ax.set_title('Bars = Avg Attendance | Line = Avg Waitlist | n = Total Events Conducted')
ax1.set_xlabel('Speaker Name')

# Sync legends
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax2.legend(h1 + h2, l1 + l2, loc='upper right')


plt.show()

# Annual Event Activity Pulse
- **Heatmap**  

In [ ]:
# Filter for events and create the pivot table
event_pulse_df = (
       combined_df.query("Type == 'Event'")
       .groupby(['Year', 'Month'])
       .size()
       .unstack(fill_value=0)
)

# Ensure all 12 months are represented even if no events happened
for month in range(1, 13):
       if month not in event_pulse_df.columns:
           event_pulse_df[month] = 0

# REVERSE the order of the years (rows) 
# This puts the most recent year (2025) at the top
event_pulse_df = event_pulse_df.iloc[::-1]

# Sort months correctly (1-12)
event_pulse_df = event_pulse_df.reindex(columns=range(1, 13))

# 3. Map month numbers to names for the x-axis
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep',
'Oct', 'Nov', 'Dec']
event_pulse_df.columns = month_names

# --- Plotting ---
fig, ax = plt.subplots(figsize=(14, 8))

sns.heatmap(
       event_pulse_df,
       annot=True,
       fmt='d',
       cmap='YlGn',
       linewidths=.5,
       cbar_kws={'label': 'Number of Events Conducted'},
       ax=ax,
       annot_kws={"size": 12, "weight": "bold"}
)

# Manually turn on ONLY the left and bottom spines
ax.spines['left'].set_visible(True)
ax.spines['bottom'].set_visible(True)

for spine in ['left', 'bottom']:
    ax.spines[spine].set_color('#292929') 
    ax.spines[spine].set_linewidth(1.0)

# Ensure ticks (both x and y) appear as per global style
ax.tick_params(axis='both', which='major', left=True, bottom=True)

fig.suptitle('Annual Event Activity Pulse', x=0.08, y=0.98, ha='left')
ax.set_title(f'Frequency of events held per month across the years | Total: {total_events}', loc='left', pad=20)

ax.set_ylabel('Year')
ax.set_xlabel('Month of Year')

# Ensure the Y-axis years are horizontal and readable
plt.yticks(rotation=0)

# Bypass "Unsupported MimeType" error in VS Code Jupyter Notebooks when using Docker
save_chart(fig, "annual_activity_pulse", CHART_PREFIX)
plt.show()